# Experiment: Generic ML vs Custom RAG

## Research Question
**"Can a model trained on generic open-source support data effectively solve our specific business tickets?"**

If the answer is **NO**, then our proposed **Hybrid RAG System** is scientifically justified.

## Methodology
1. **Create Generic Dataset**: Simulate standard open-source tech support data (e.g., Kaggle/HuggingFace style data).
2. **Train Generic Model**: Train a LinearSVC on this external data.
3. **Zero-Shot Test**: Apply this model to our specific `customer_support_tickets.csv`.
4. **Analyze Failure Cases**: Show where it lacks the business context that RAG provides.

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

# ---------------------------------------------------------
# 1. SIMULATE OPEN SOURCE DATA (Generic Tech Support)
# ---------------------------------------------------------
# Imagine this comes from a standard dataset like 'tech-qa-corpus'
generic_data = {
    'text': [
        "My internet is slow and pages won't load",
        "I forgot my password and cannot login",
        "The screen is black and won't turn on",
        "I want to cancel my subscription",
        "Where can I download the driver?",
        "My mouse is not working",
        "Blue screen of death on startup",
        "How do I update my profile picture?",
        "I was charged twice for the same item",
        "The battery drains very fast",
        "Wifi connects but no internet access",
        "Reset my account pincode",
        "Cancel my monthly plan please",
        "Refund my last payment",
        "Product arrived damaged"
    ] * 100,  # Duplicate to simulate volume
    'action': [
        "Troubleshoot Network",
        "Reset Password",
        "Hardware Diagnostic",
        "Process Cancellation",
        "Software Update",
        "Hardware Diagnostic",
        "System Repair",
        "Account Settings",
        "Billing Dispute",
        "Hardware Diagnostic",
        "Troubleshoot Network",
        "Reset Password",
        "Process Cancellation",
        "Process Refund",
        "Return Merchandise"
    ] * 100
}

df_generic = pd.DataFrame(generic_data)
print(f"Generic Dataset Created: {df_generic.shape[0]} rows")
print(df_generic.head())

Generic Dataset Created: 1500 rows
                                       text                action
0  My internet is slow and pages won't load  Troubleshoot Network
1     I forgot my password and cannot login        Reset Password
2     The screen is black and won't turn on   Hardware Diagnostic
3          I want to cancel my subscription  Process Cancellation
4          Where can I download the driver?       Software Update


## Train the "Generic Model"

In [2]:
# Train a pipeline on this generic data
generic_model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2))),
    ('clf', LinearSVC())
])

generic_model.fit(df_generic['text'], df_generic['action'])
print("Generic Model Trained successfully on 'Open Source' data.")

Generic Model Trained successfully on 'Open Source' data.


## Test on OUR Specific Data
Now we see if this generic model can handle our specific tickets.

In [3]:
# Load our specific business data
df_specific = pd.read_csv('../data/raw/customer_support_tickets.csv')

# Create 'combined_text' again
df_specific['combined_text'] = (df_specific['Ticket Subject'].fillna('') + " " + df_specific['Ticket Description'].fillna('')).str.lower()

# Define the 'True' labels (What we mapped earlier)
action_map = {
    'Technical issue': 'Troubleshoot Network',  # Best guess mapping
    'Billing inquiry': 'Billing Dispute',
    'Cancellation request': 'Process Cancellation',
    'Product inquiry': 'Hardware Diagnostic', # Mismatch potential
    'Refund request': 'Process Refund'
}
df_specific['True_Label'] = df_specific['Ticket Type'].map(action_map)
df_specific.dropna(subset=['True_Label'], inplace=True)

# PREDICT using Generic Model
df_specific['Generic_Prediction'] = generic_model.predict(df_specific['combined_text'])

# Check Accuracy
print("--- Performance of Generic Model on Specific Data ---")
print(classification_report(df_specific['True_Label'], df_specific['Generic_Prediction']))

--- Performance of Generic Model on Specific Data ---
                      precision    recall  f1-score   support

    Account Settings       0.00      0.00      0.00         0
     Billing Dispute       0.20      0.05      0.08      1634
 Hardware Diagnostic       0.19      0.63      0.30      1641
Process Cancellation       0.19      0.19      0.19      1695
      Process Refund       0.26      0.00      0.01      1752
      Reset Password       0.00      0.00      0.00         0
  Return Merchandise       0.00      0.00      0.00         0
     Software Update       0.00      0.00      0.00         0
       System Repair       0.00      0.00      0.00         0
Troubleshoot Network       0.16      0.01      0.02      1747

            accuracy                           0.17      8469
           macro avg       0.10      0.09      0.06      8469
        weighted avg       0.20      0.17      0.12      8469



c:\Users\SRINATH\Desktop\data science\machine learing\ml project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\SRINATH\Desktop\data science\machine learing\ml project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\SRINATH\Desktop\data science\machine learing\ml project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this b

## Failure Analysis (The "Why RAG?")
Let's look at examples where the Generic Model failed but RAG would succeed.

In [4]:
failures = df_specific[df_specific['True_Label'] != df_specific['Generic_Prediction']].sample(5)

print(f"{'Ticket Subject':<50} | {'Generic Model Thought':<25} | {'Actual Need':<25}")
print("-" * 110)

for _, row in failures.iterrows():
    print(f"{row['Ticket Subject'][:47]:<50} | {row['Generic_Prediction']:<25} | {row['True_Label']:<25}")

Ticket Subject                                     | Generic Model Thought     | Actual Need              
--------------------------------------------------------------------------------------------------------------
Delivery problem                                   | Hardware Diagnostic       | Troubleshoot Network     
Data loss                                          | Hardware Diagnostic       | Troubleshoot Network     
Battery life                                       | Hardware Diagnostic       | Billing Dispute          
Refund request                                     | Hardware Diagnostic       | Troubleshoot Network     
Data loss                                          | Hardware Diagnostic       | Process Refund           
